# Pertemuan 3 - Data Cleaning: Missing, Outlier & Ekstraksi
**Nama:** RANU RATMAJA
**NIM:** 230401010104
**Prodi:** Sistem Informasi
**Universitas:** Universitas Siber Asia (UNSIA)
**Tahun:** 2023

## Tujuan Praktikum
Pada pertemuan ini mahasiswa mempelajari teknik **Data Cleaning** yang meliputi: penanganan missing values, deteksi dan penanganan outlier menggunakan metode IQR dan Z-Score, serta teknik ekstraksi fitur dari data mentah.

## 1. Import Library

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
print('Library berhasil diimport!')

## 2. Membuat Dataset dengan Missing Values & Outlier

In [2]:
np.random.seed(42)
data = {
 'Nama': ['Andi','Budi','Citra','Dian','Eka','Fajar','Gita','Hana','Ivan','Joko'],
 'Usia': [22, 25, None, 23, 28, 21, None, 24, 26, 200],
 'Tinggi_cm': [165, 170, 168, None, 175, 160, 172, 169, 500, 173],
 'Berat_kg': [55, 68, 60, 62, None, 50, 65, 58, 70, 63],
 'IPK': [3.5, 3.8, 3.2, None, 3.9, 2.8, 3.6, 3.4, 3.7, None]
}
df = pd.DataFrame(data)
print('Dataset awal:')
print(df)

## 3. Identifikasi Missing Values

In [3]:
print('Jumlah Missing Values per Kolom:')
print(df.isnull().sum())
print('\nPersentase Missing Values:')
print((df.isnull().sum() / len(df) * 100).round(2), '%')
plt.figure(figsize=(8,4))
sns.heatmap(df.isnull(), cbar=False, cmap='viridis', yticklabels=False)
plt.title('Peta Missing Values (kuning = missing)')
plt.show()

## 4. Penanganan Missing Values

In [4]:
df_clean = df.copy()
# Isi dengan median untuk kolom numerik
for col in ['Usia', 'Tinggi_cm', 'Berat_kg', 'IPK']:
 median_val = df_clean[col].median()
 df_clean[col].fillna(median_val, inplace=True)
 print(f'Kolom {col}: missing diisi dengan median = {median_val}')
print('\nMissing values setelah penanganan:')
print(df_clean.isnull().sum())

## 5. Deteksi Outlier dengan Metode IQR

In [5]:
def detect_outlier_iqr(df, col):
 Q1 = df[col].quantile(0.25)
 Q3 = df[col].quantile(0.75)
 IQR = Q3 - Q1
 lower = Q1 - 1.5 * IQR
 upper = Q3 + 1.5 * IQR
 outliers = df[(df[col] < lower) | (df[col] > upper)]
 return outliers, lower, upper

for col in ['Usia', 'Tinggi_cm', 'Berat_kg']:
 outliers, low, up = detect_outlier_iqr(df_clean, col)
 print(f'Kolom {col}: batas [{low:.1f}, {up:.1f}] | Outlier: {len(outliers)} baris')

## 6. Visualisasi Outlier dengan Boxplot

In [6]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
cols = ['Usia', 'Tinggi_cm', 'Berat_kg']
colors = ['steelblue', 'coral', 'mediumseagreen']
for ax, col, color in zip(axes, cols, colors):
 ax.boxplot(df_clean[col], patch_artist=True,
 boxprops=dict(facecolor=color, alpha=0.7))
 ax.set_title(f'Boxplot {col}')
 ax.set_ylabel(col)
plt.suptitle('Deteksi Outlier dengan Boxplot', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Penanganan Outlier - Capping (Winsorizing)

In [7]:
df_no_outlier = df_clean.copy()
for col in ['Usia', 'Tinggi_cm', 'Berat_kg']:
 Q1 = df_no_outlier[col].quantile(0.25)
 Q3 = df_no_outlier[col].quantile(0.75)
 IQR = Q3 - Q1
 lower = Q1 - 1.5 * IQR
 upper = Q3 + 1.5 * IQR
 df_no_outlier[col] = df_no_outlier[col].clip(lower=lower, upper=upper)
print('Dataset setelah outlier di-cap:')
print(df_no_outlier[['Nama','Usia','Tinggi_cm','Berat_kg']])

## 8. Ekstraksi Fitur

In [8]:
# Ekstraksi fitur: BMI dan kategori BMI
df_no_outlier['BMI'] = (df_no_outlier['Berat_kg'] / (df_no_outlier['Tinggi_cm']/100)**2).round(2)
df_no_outlier['Kategori_BMI'] = pd.cut(df_no_outlier['BMI'],
 bins=[0, 18.5, 24.9, 29.9, 100],
 labels=['Kurus', 'Normal', 'Overweight', 'Obesitas'])
df_no_outlier['Kategori_IPK'] = pd.cut(df_no_outlier['IPK'],
 bins=[0, 2.5, 3.0, 3.5, 4.0],
 labels=['Cukup', 'Memuaskan', 'Sangat Memuaskan', 'Cumlaude'])
print('Dataset dengan fitur baru (BMI & Kategori):')
print(df_no_outlier[['Nama','BMI','Kategori_BMI','IPK','Kategori_IPK']])

## Kesimpulan
- Missing values dapat diidentifikasi dengan `isnull()` dan divisualisasikan menggunakan heatmap.
- Penanganan missing values dilakukan dengan mengisi menggunakan nilai median agar tidak terpengaruh outlier.
- Outlier dideteksi menggunakan metode IQR (Interquartile Range) dan divisualisasikan dengan boxplot.
- Capping (Winsorizing) adalah teknik penanganan outlier dengan membatasi nilai pada batas atas/bawah IQR.
- Ekstraksi fitur (BMI dan kategori) menghasilkan informasi baru yang lebih bermakna dari data mentah.